<a href="https://colab.research.google.com/github/Yuzefita/ml_homeworks_pavluchuk/blob/main/hw03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Домашнее задание 3 KNN


In [2]:
from sklearn.datasets import load_iris
import pandas as pd
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


Анализ данных:
*   арзмер
*   тип данных
*   базовые статистики
*   визуализация








In [4]:
print(f"Размер:{df.shape}")
df.info()
df.describe()

Размер:(150, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


баланс классов (в нашем случае идеально сбалансированы)

In [6]:

print(df['target'].value_counts(normalize=True))

target
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64


# Подготовка данных

обработка пропусков
(пропусков нет, видно из info())

разделение на train, test:

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis = 1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size =0.2,random_state =42)

масштабирование признаков

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

почему масштабирование важно для KNN: тк KNN использует метрики расстояния, то если признаки будут иметь разные диапазоны значений
признаки с большими модулями будут доминировать в формуле расстояния, фактически заставляя модель игнорировать остальные данные

Почему нельзя подбирать параметры на тестовой выборке? Модель просто обучится для конкретного теста и то есть запомнила решение для конкретного тест но не выучила общие правила

# Обучение KNN
разные значения n_neighbors
:

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
for k in [1, 3, 5, 11, 21, 50]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    print(f"k = {k:2d} | Accuracy: {accuracy_score(y_test, pred):.4f}")

k =  1 | Accuracy: 1.0000
k =  3 | Accuracy: 1.0000
k =  5 | Accuracy: 1.0000
k = 11 | Accuracy: 1.0000
k = 21 | Accuracy: 1.0000
k = 50 | Accuracy: 0.9333


In [14]:
for weight in ['uniform', 'distance']:
    model = KNeighborsClassifier(n_neighbors=5, weights=weight)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"Weights: {weight:8s}  Accuracy: {acc:.4f}")

Weights: uniform   Accuracy: 1.0000
Weights: distance  Accuracy: 1.0000


In [15]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_train_scaled, y_train)
print("Лучшие параметры:", grid_search.best_params_)
print("Лучшая точность на кросс-валидации:", grid_search.best_score_)

Лучшие параметры: {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'distance'}
Лучшая точность на кросс-валидации: 0.9583333333333334
